## Primer acercamiento 

Hice un primer acercamiento bastante interesante para saber que es lo que esta pasando o que es lo que tenemos con el DS. 

In [13]:
import pandas as pd

RUTA = '../data/raw/2025O3.xls'

df_crudo = pd.read_excel(RUTA, header=None, nrows=8)
print(df_crudo.to_string())

                    0     1    2    3    4    5    6    7    8    9    10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29   30   31   32   33   34   35   36   37
0                FECHA  HORA  ACO  AJM  AJU  ATI  BJU  CAM  CCA  CHO  COY  CUA  CUT  FAC  FAR  GAM  HGM  INN  IZT  LLA  LPR  MER  MGH  MON  MPA  NEZ  PED  SAC  SAG  SFE  SJA  TAH  TLA  TLI  UAX  UIZ  VIF  XAL
1  2025-01-01 00:00:00     1  -99   46   25   26  -99   12   28   10  -99   29   18   29  -99  -99   17   44   12   23   11   11   31   11  -99    2   34    6  -99  -99  -99   27   19   20    4    7    2  -99
2  2025-01-01 00:00:00     2  -99   47   19   36  -99    7   24    7  -99   28   13   24  -99  -99   25   46   14   11    3   13   23   10  -99    4   29    7  -99  -99  -99   17   22   15    4    8    2  -99
3  2025-01-01 00:00:00     3  -99   47   19   35  -99    2   13    6  -99   22   14    9  -99  -99   14   45    7    3    2   10    4    2  -99    2   26    7  -99 

## Resultados del primer acercamiento 

Al cargar apenas unos datos ya pude saber como esta constituida el csv o el DS. 

Al verlo puedo ver la fecha que constituye de el año, mes y dia, para consecuentemente ver la hora ( del 1 al 24) junto con los datos de cada sitio o sensor que se tiene en la CDMX. 

Inicialmente ya puedo detectar valores interesantes como valores increiblemente bajos como 2 o 1, o hasta valores increimblemente altos o directamente fuera de sentido como el -99.

El -99 implica que indica que:
* El sensor esta fuera de servicio.
* Esta en modo de mantenimiento. 
* No tiene corriente electrica.

## Celda 2

La intencion de esta celda es cuantificar datos, ver que existe y saber cuanto daño hace a nuestro potencial analisis. 

In [14]:
df = pd.read_excel(RUTA, header=0)

print("Dimensiones:", df.shape)
print("\nColumnas:", list(df.columns))
print("\nTipos de dato:")
print(df.dtypes.value_counts())

Dimensiones: (8760, 38)

Columnas: ['FECHA', 'HORA', 'ACO', 'AJM', 'AJU', 'ATI', 'BJU', 'CAM', 'CCA', 'CHO', 'COY', 'CUA', 'CUT', 'FAC', 'FAR', 'GAM', 'HGM', 'INN', 'IZT', 'LLA', 'LPR', 'MER', 'MGH', 'MON', 'MPA', 'NEZ', 'PED', 'SAC', 'SAG', 'SFE', 'SJA', 'TAH', 'TLA', 'TLI', 'UAX', 'UIZ', 'VIF', 'XAL']

Tipos de dato:
int64             37
datetime64[us]     1
Name: count, dtype: int64


## Descubrimientos de la celda 2

Claude me recomendo ver si es que tenemos algun potencial error en los datos o como estan etiquetados. Puedo ver que tengo int64 que potencialmente pueden ser los valores de cada estcion. 

## Celda 3: cuantificar el centinela

En realidad la idea es poder ver todas las estaciones como un solo vector, ademas podermos ver de manera mas facil los valores que tenemos en nuestro DS para poder empezar a trabajar con algunos datos.

In [15]:
estaciones = [c for c in df.columns if c not in ('FECHA', 'HORA')]

valores = df[estaciones].values.ravel()

print("Mínimo absoluto:", valores.min())
print("Máximo absoluto:", valores.max())
print("\nValores más frecuentes:")
print(pd.Series(valores).value_counts().head(5))

Mínimo absoluto: -99
Máximo absoluto: 168

Valores más frecuentes:
-99    96537
 2     10314
 1      7836
 3      7273
 4      5488
Name: count, dtype: int64


## Descubrimientos de la celda 3 

Como se pudo observar esta cuantificacion si tenemos registrados numeros de hasta -99 por el potencial error en los mismo, 

Pero si podemmos detectar la frecuencia y en que tiempo se hizo la muestra y ver si conincide con nuestros conocimiento.

## Celda 4.  El daño concreto de tener datos fuera de lugar.

Es importante denotar que tenemos datos atipicos o fuera de rango en su totalidad 

¿Porque? Es claro, en la celda 3 vimos que tenemos valores de -99 esto es fisicamente imposible, literalmente no podemos tener ozono negativo por lo que es claro que tenemos una flag ahi o un valor qeu quiere indicar una potencial falla o similares.

In [16]:
import numpy as np

n_total = valores.size
n_flag  = (valores == -99).sum()

print(f"Celdas totales:   {n_total:,}")
print(f"Celdas con -99:   {n_flag:,}  ({n_flag/n_total*100:.2f}%)")
print()
print(f"Media SIN limpiar: {valores.mean():.2f} ppb")

limpio = np.where(valores == -99, np.nan, valores)
print(f"Media limpia:      {np.nanmean(limpio):.2f} ppb")
print(f"Sesgo introducido: {valores.mean() - np.nanmean(limpio):.2f} ppb")

Celdas totales:   315,360
Celdas con -99:   96,537  (30.61%)

Media SIN limpiar: -7.75 ppb
Media limpia:      32.51 ppb
Sesgo introducido: -40.26 ppb


## Descubrimiento de la celda 4

Con esto podemos darnos cuenta que tan mal esta afectando nuesto DS ese "-99" literalmente tenemos medias extremadamente irreales (no podemos tener concentraciones de ozono negativas)

Pero ya limpio tenemos valores esperados o que si podemos tener de estos valores.

## Celda 5 Cobertura por estacion. 

Lo que se va a intentar hacer en esta celda es no remplazar los "-99" por valores reales si no mejor ponerlos como nan para limpiar el DS y aproximarnos a numeros reales (Ademas de trabajar con un DS real o sin errores).



In [17]:
import numpy as np

df_nan = df.copy()
df_nan[estaciones] = df_nan[estaciones].replace(-99, np.nan)

cobertura = df_nan[estaciones].notna().mean().sort_values(ascending=False)

print("Cobertura (% de horas con dato válido):")
print((cobertura * 100).round(2).to_string())

Cobertura (% de horas con dato válido):
CCA    96.93
HGM    93.20
PED    93.12
CUT    92.40
TLA    92.25
CAM    90.45
MGH    90.39
SAG    89.78
LLA    89.03
AJM    88.64
LPR    88.63
UIZ    88.09
TLI    87.51
VIF    87.36
INN    86.83
ATI    86.59
FAC    86.38
MER    84.17
UAX    77.58
NEZ    73.64
GAM    73.29
FAR    70.89
MPA    65.39
ACO    65.18
TAH    63.74
XAL    60.64
MON    60.51
SAC    55.55
BJU    54.84
IZT    53.22
CUA    49.78
CHO    34.34
AJU    27.65
COY     0.00
SFE     0.00
SJA     0.00


## Conclusion a los datos de la celda 5

No tenemos ninguna estacion al 100% de su operabilidad, algo completamente esperado. 

Pero diria que casi el 75% de las estaciones nos pueden dar datos utiles, si bien tenemos estaciones que deben ser ignoradas o mas bien tomarlas en cuneta por valores inferiores al 60% todas nos puedes servir. 

Excepciones tengo 3 estaciones que directamente estan desconectadas. Si es que tenemos 3 estaciones que el 100% de su tiempo no registraron nada, podriamos considerar eliminarlas del DS porque etas afectan enormemente que tanto aprace nuestro "-99" 

Algo interesante una estación con 50% de cobertura sigue aportando 4,380 horas reales, y en alguna de esas horas pudo haber sido la que dominó el índice.

Gran ejemplo es la de CUA o Cuajimalpa, esta tiene **49.78%** de cobertura, pero ironicamente en enero del 2026 resitro los 160ppb activando una contingencia Fase 1 en la ZMVM

## Celda 5.1 

Como veniamos diciendo necesitamos ver que tanta racha tienen la celdas 

In [18]:
def racha_max(serie):
    """Máximo número de horas consecutivas sin dato."""
    falta = serie.isna().astype(int)
    grupos = (falta != falta.shift()).cumsum()
    return falta.groupby(grupos).sum().max()

rachas = df_nan[estaciones].apply(racha_max).sort_values()

resumen = pd.DataFrame({
    'cobertura_%': (cobertura * 100).round(2),
    'racha_max_h': rachas,
    'racha_max_dias': (rachas / 24).round(1),
}).sort_values('cobertura_%', ascending=False)

print(resumen.to_string())

     cobertura_%  racha_max_h  racha_max_dias
CCA        96.93           17             0.7
HGM        93.20          231             9.6
PED        93.12          341            14.2
CUT        92.40          181             7.5
TLA        92.25          134             5.6
CAM        90.45          464            19.3
MGH        90.39          231             9.6
SAG        89.78          304            12.7
LLA        89.03          427            17.8
AJM        88.64          504            21.0
LPR        88.63          725            30.2
UIZ        88.09          195             8.1
TLI        87.51          576            24.0
VIF        87.36          479            20.0
INN        86.83          432            18.0
ATI        86.59          394            16.4
FAC        86.38          687            28.6
MER        84.17         1021            42.5
UAX        77.58          882            36.8
NEZ        73.64         1303            54.3
GAM        73.29         1139     

## Conclusiones de la celda 5.1

En la celda 5 identificamos estaciones con disponibilidad alta. Pero como
los datos se procesarán como serie temporal, surge una pregunta que la
cobertura no responde: ¿qué pasa si una estación con buena cobertura tiene
contacto cero durante un día, dos, cinco o incluso una semana seguida?

Por eso es necesario revisar dos criterios: cobertura, y racha máxima
(el bloque continuo más largo sin servicio).

### Hallazgo: la cobertura por sí sola es un criterio insuficiente

| Estación | Cobertura | Racha máx |
|---|---|---|
| CAM | 90.45% | 464 h (19.3 días) |
| UIZ | 88.09% | 195 h (8.1 días) |

CAM tiene mejor cobertura que UIZ, pero estuvo fuera de operación 19 días
seguidos frente a 8. Para análisis de series temporales UIZ es la mejor
opción, algo que el porcentaje de cobertura oculta por completo.

### Estación piloto: CCA

| Posición | Estación | Cobertura | Racha máx |
|---|---|---|---|
| 1 | CCA | 96.93% | 17 h |
| 2 | CUT | 92.40% | 181 h |
| 3 | TLA | 92.25% | 134 h |

CCA gana en ambos criterios simultáneamente, sin trade-off. La diferencia
con la siguiente estación es de casi un orden de magnitud en racha máxima
(17 h contra 134 h): es la única estación de la red que en 2025 no
registró ningún periodo prolongado fuera de operación.

Nota: 17 h es el bloque continuo más largo, no el total de faltantes.
CCA tiene ~269 horas faltantes en total, repartidas en varios bloques.
La distribución de esos bloques se analiza en la siguiente celda.

## Celda 6: distribución de bloques faltantes en CCA

Esta celda esta creada apra ver como se reparten las 269 horas faltantes de esta estacion piloto 

Cómo funciona rachas_todas 

Es el mismo patrón de racha_max pero devolviendo todos los bloques en vez de solo el mayor. Paso a paso:

* `serie.isna().astype(int)` → convierte a 1 donde falta, 0 donde hay dato
* `falta != falta.shift()` → True en cada cambio de estado (de dato a hueco o viceversa)
* `.cumsum()` → asigna un identificador único a cada bloque contiguo del mismo estado
* `.groupby(grupos).sum()` → los bloques de faltantes suman su longitud; los de datos suman 0
* `r[r > 0]` → descarta los bloques de datos, deja solo los huecos

In [19]:
def rachas_todas(serie):
    """Devuelve las longitudes de todos los bloques de faltantes."""
    falta = serie.isna().astype(int)
    grupos = (falta != falta.shift()).cumsum()
    r = falta.groupby(grupos).sum()
    return r[r > 0]

r_cca = rachas_todas(df_nan['CCA'])

print(f"Número de bloques:        {len(r_cca)}")
print(f"Horas faltantes totales:  {r_cca.sum()}")
print(f"Bloque más largo:         {r_cca.max()} h")
print(f"Bloque más corto:         {r_cca.min()} h")
print(f"Mediana:                  {r_cca.median()} h")
print("\nDistribución de longitudes:")
print(r_cca.value_counts().sort_index().to_string())

Número de bloques:        70
Horas faltantes totales:  269
Bloque más largo:         17 h
Bloque más corto:         2 h
Mediana:                  3.0 h

Distribución de longitudes:
CCA
2      1
3     58
4      5
5      1
10     1
13     1
14     2
17     1


## Conclusiones de la celda 6

Es superinteresante que tenemos **58 bloques** de 3 horas faltantes, esto es potencialmente un patron, pueden ser desde fallas de energia pero tambien pausas de calibracion o mantenimiento, porque si fueran totalmente aleatorios tendriamos bloques sin sentido de duraciones largas. En el caso de aqui es increible que el pico de este caso es de 58 bloques de 3 hroas. Por lo que se toma la decision de ir a una celda 6.1 para ver si ese patron es de ciertas horas. 


## Celda 6.1 verificacion de patrones en los bloques de no operacion 

In [20]:
falt_cca = df_nan[df_nan['CCA'].isna()]

print("Faltantes por hora del día:")
print(falt_cca['HORA'].value_counts().sort_index().to_string())

Faltantes por hora del día:
HORA
1     61
2     61
3     61
4      3
5      3
6      3
7      3
8      3
9      3
10     4
11     7
12    11
13    12
14    10
15     6
16     2
17     2
18     2
19     2
20     2
21     2
22     2
23     2
24     2


## Conclusiones de la celda 6

La celda 5.1 mostró que CCA tiene una racha máxima de 17 h. Faltaba saber
cómo se reparten las 269 horas faltantes: si son muchos huecos cortos o
pocos huecos largos. Esa distinción determina la estrategia de tratamiento.

### Los faltantes no son aleatorios

| Longitud del bloque | Ocurrencias |
|---|---|
| 2 h | 1 |
| 3 h | 58 |
| 4 h | 5 |
| 5 h | 1 |
| 10 h | 1 |
| 13 h | 1 |
| 14 h | 2 |
| 17 h | 1 |

70 bloques, 269 horas faltantes en total.

El 83% de los bloques mide exactamente 3 horas, y **no existe ningún bloque
de 1 hora**. Una distribución de fallas aleatorias (cortes de energía,
errores de transmisión) produciría muchos huecos de 1 h y una curva
decreciente. Se observa lo contrario.

Esto indica un proceso programado, no fallas. La hipótesis más probable es
calibración automática del analizador de ozono: los equipos de monitoreo
realizan verificaciones periódicas de cero y span durante las cuales no
reportan concentración ambiental, con duración fija.

### El proceso ocurre de madrugada

Distribución de faltantes por hora del día:

| Hora | Faltantes |
|---|---|
| 1 | 61 |
| 2 | 61 |
| 3 | 61 |
| 4–10 | 3–4 |
| 11–15 | 6–12 |
| 16–24 | 2 |

Las horas 1, 2 y 3 registran exactamente 61 faltantes cada una. Ninguna otra
hora se aproxima. Son 61 eventos de 3 horas continuas entre las 00:00 y las
02:59, aproximadamente cinco veces por mes.

### Implicación: la interpolación es segura en este caso

El ozono troposférico se forma por reacción fotoquímica y requiere radiación
solar. De madrugada no hay formación y las concentraciones son bajas y
estables: es la zona más plana del ciclo diurno.

Que los faltantes se concentren ahí tiene tres consecuencias favorables:

1. Interpolar sobre una zona plana introduce error mínimo. Si el patrón
   hubiera sido vespertino, se estarían inventando picos.
2. No hay sesgo respecto a los umbrales de la NOM-172: el primer corte
   (58 ppb) no se alcanza de madrugada, por lo que no se pierden lecturas
   que habrían cruzado bandas.
3. La métrica de daño no se ve afectada por el tratamiento de faltantes.

Se observa un pico secundario en las horas 11–15, por encima del ruido de
fondo. Corresponde presumiblemente a los bloques largos (10–17 h), que
recibirán tratamiento distinto.

### Criterio de tratamiento adoptado

| Longitud | Bloques | Horas | % de faltantes | Tratamiento |
|---|---|---|---|---|
| 2–5 h | 65 | 201 | 74.7% | Interpolación |
| 10–17 h | 5 | 68 | 25.3% | Segmentación de la serie |

No existe ningún bloque entre 5 h y 10 h. Ese vacío proporciona el punto de
corte sin necesidad de fijar un umbral arbitrario.

### Advertencia metodológica

La interpolación produce valores artificialmente suaves respecto a sus
vecinos, y el detector de Capa 2 se calibra precisamente sobre
|x[t] − x[t−1]|. Con 201 horas interpoladas sobre 8,760 (2.3%) el efecto es
pequeño, pero no nulo.

Decisión: marcar las horas interpoladas con una bandera booleana, de modo
que puedan excluirse del cálculo de umbral si se detecta sesgo.

## Celda 7: la distribución de CCA respecto a los umbrales

In [21]:
BANDAS_O3 = [58, 90, 135, 175]  # ppb — NOM-172-2023, Tabla 6
NOMBRES = ['Buena', 'Aceptable', 'Mala', 'Muy Mala', 'Ext. Mala']

cca = df_nan['CCA'].dropna()

print(f"Lecturas válidas: {len(cca)}")
print(f"Mín: {cca.min():.0f}   Máx: {cca.max():.0f}   Media: {cca.mean():.2f}")
print(f"Percentiles — p50: {cca.quantile(.50):.0f}  "
      f"p90: {cca.quantile(.90):.0f}  "
      f"p99: {cca.quantile(.99):.0f}")

print("\nDistribución de bandas:")
banda = np.searchsorted(BANDAS_O3, cca, side='left')
for i, nombre in enumerate(NOMBRES):
    n = (banda == i).sum()
    print(f"  {nombre:<12} {n:>6}  ({n/len(cca)*100:>6.3f}%)")

Lecturas válidas: 8491
Mín: 0   Máx: 149   Media: 35.44
Percentiles — p50: 26  p90: 82  p99: 121

Distribución de bandas:
  Buena          6578  (77.470%)
  Aceptable      1289  (15.181%)
  Mala            604  ( 7.113%)
  Muy Mala         20  ( 0.236%)
  Ext. Mala         0  ( 0.000%)


In [22]:
print("Lecturas dentro de N ppb por DEBAJO de cada umbral:\n")
print(f"{'':>6}" + "".join(f"{'≤'+str(d)+' ppb':>12}" for d in [2, 4, 8, 16, 32]))

for tau in BANDAS_O3:
    fila = f"τ={tau:<4}"
    for d in [2, 4, 8, 16, 32]:
        n = ((cca > tau - d) & (cca <= tau)).sum()
        fila += f"{n:>6} ({n/len(cca)*100:>4.1f}%)"
    print(fila)

Lecturas dentro de N ppb por DEBAJO de cada umbral:

            ≤2 ppb      ≤4 ppb      ≤8 ppb     ≤16 ppb     ≤32 ppb
τ=58      83 ( 1.0%)   188 ( 2.2%)   427 ( 5.0%)   887 (10.4%)  2251 (26.5%)
τ=90      38 ( 0.4%)   100 ( 1.2%)   225 ( 2.6%)   545 ( 6.4%)  1289 (15.2%)
τ=135      3 ( 0.0%)    12 ( 0.1%)    22 ( 0.3%)    74 ( 0.9%)   294 ( 3.5%)
τ=175      0 ( 0.0%)     0 ( 0.0%)     0 ( 0.0%)     0 ( 0.0%)     5 ( 0.1%)


## Conclusiones de la celda 7

Las celdas anteriores analizaron la disponibilidad de datos. Ésta es la
primera que analiza su contenido, y responde si el ataque es viable sobre
la estación seleccionada.

### Perfil de CCA

8,491 lecturas válidas. Mín 0 ppb, máx 149 ppb, media 35.44 ppb.
Percentiles: p50 = 26, p90 = 82, p99 = 121.

| Banda | Lecturas | CCA | Red completa |
|---|---|---|---|
| Buena | 6,578 | 77.470% | 81.257% |
| Aceptable | 1,289 | 15.181% | 14.425% |
| Mala | 604 | **7.113%** | 4.174% |
| Muy Mala | 20 | 0.236% | 0.144% |
| Ext. Mala | 0 | 0.000% | 0.000% |

CCA registra 1.7 veces más horas en banda "Mala" que el promedio de la red.
Es consistente con su ubicación en el sur de la ciudad, zona de acumulación
de ozono por transporte de precursores y radiación intensa.

**No existe conflicto entre criterios de selección.** La estación con mejor
calidad de datos resulta ser también de las más expuestas. Era un riesgo
que esta celda debía descartar.

### Ventana de oportunidad por umbral

Lecturas situadas por debajo de cada umbral, a distancia menor o igual a N:

| τ (ppb) | ≤8 ppb | ≤16 ppb | ≤32 ppb |
|---|---|---|---|
| 58 | 5.0% | 10.4% | 26.5% |
| 90 | 2.6% | 6.4% | 15.2% |
| 135 | 0.3% | 0.9% | 3.5% |
| 175 | 0.0% | 0.0% | 0.1% |

Los dos primeros umbrales concentran la superficie de ataque. Un
desplazamiento de 16 ppb encuentra el 10.4% de las lecturas en posición de
cruzar el corte de 58 ppb; uno de 32 ppb alcanza el 26.5%.

**Ambos desplazamientos están dentro de la variabilidad legítima horaria del
ozono.** El ataque capaz de cambiar la decisión no requiere producir un
valor anómalo. Esta observación es la justificación central del trabajo de
detección basado en aprendizaje: un detector de valores extremos no
capturaría estos casos.

### τ = 175 queda fuera de alcance

El máximo registrado en CCA durante 2025 es 149 ppb. No existe ninguna
lectura dentro de 32 ppb del umbral de 175. Se excluye del análisis por
ausencia de superficie.

### Validación empírica de la elección de f

El umbral de contingencia del PPRCAA (~154 ppb) tampoco se alcanza en CCA
durante 2025. De haberse adoptado el PPRCAA como función de decisión, la
estación piloto no registraría ningún evento de daño posible.

La decisión de adoptar la NOM-172 queda respaldada por evidencia empírica,
no sólo por argumento de disponibilidad normativa.

### Ajuste pendiente

La asimetría inflar/desinflar calculada sobre la red completa es menos
pronunciada en CCA (77.47% en "Buena" frente a 81.26%). Debe recalcularse
específicamente para la estación piloto en las tablas finales.

### Coherencia entre celdas

8,491 lecturas válidas + 269 faltantes = 8,760 horas. Consistente con la
cobertura de 96.93% (celda 5) y el conteo de faltantes (celda 6).

## Celda 8 argumentos del Dr Jaime 
 
Es respecto al dibujo que me hizo el doc jaime de alterar los datos a lo largo del tiempo. 

In [23]:
perfil = df_nan.groupby('HORA')['CCA'].agg(
    media='mean', std='std', p50='median', p90=lambda s: s.quantile(.90), max='max'
)
print(perfil.round(2).to_string())

      media    std   p50    p90    max
HORA                                  
1     16.09  11.22  15.0   32.0   46.0
2     15.48  10.64  15.0   30.0   45.0
3     14.95  10.37  14.0   29.0   44.0
4     13.54   9.80  13.0   28.0   51.0
5     12.46   8.88  11.0   25.0   44.0
6      9.96   7.74   8.0   21.0   37.0
7      6.24   5.79   4.0   15.0   32.0
8      7.66   6.07   6.0   17.0   30.0
9     15.90   9.27  14.0   29.0   44.0
10    28.80  12.46  27.0   47.0   67.0
11    44.60  15.84  44.0   67.0   85.0
12    60.46  18.52  62.0   85.0  107.0
13    73.05  21.21  74.0   98.8  124.0
14    81.46  24.17  84.0  111.0  138.0
15    82.58  27.01  84.0  115.0  144.0
16    77.86  28.69  77.0  113.8  147.0
17    69.47  28.75  67.0  108.0  149.0
18    57.40  24.06  55.0   91.0  129.0
19    42.95  18.25  43.0   66.8   99.0
20    32.82  16.47  32.0   55.0   75.0
21    25.48  14.09  26.0   44.0   65.0
22    20.47  12.90  20.0   38.8   61.0
23    17.62  12.12  16.0   35.0   52.0
24    16.39  11.33  15.0 

## Conclusiones de la celda 8

Las celdas anteriores caracterizaron la disponibilidad y la distribución de
los datos. Ésta caracteriza su **estructura temporal**, que es la base sobre
la que operará la detección con memoria (Capa 3).

### El ciclo diurno está bien definido

| Hora | Media | std | p90 | Máx |
|---|---|---|---|---|
| 7 | 6.24 | 5.79 | 15.0 | 32.0 |
| 12 | 60.46 | 18.52 | 85.0 | 107.0 |
| 15 | 82.58 | 27.01 | 115.0 | 144.0 |
| 17 | 69.47 | 28.75 | 108.0 | 149.0 |
| 22 | 20.47 | 12.90 | 38.8 | 61.0 |

Mínimo a las 7 h (6.24 ppb), máximo a las 15 h (82.58 ppb): un factor de 13×
entre ambos extremos.

El comportamiento corresponde al esperado para ozono troposférico, que se
forma por reacción fotoquímica y requiere radiación solar. Existe por tanto
estructura temporal suficiente para que un modelo con memoria detecte
desviaciones de forma, no sólo de magnitud.

### La variabilidad legítima escala con la hora

| Hora | std | Un flip de 16 ppb equivale a |
|---|---|---|
| 7 | 5.79 | 2.76 desviaciones |
| 12 | 18.52 | 0.86 desviaciones |
| 17 | 28.75 | 0.56 desviaciones |

La desviación estándar varía en un factor de 5 a lo largo del día. Dado que
la variabilidad legítima constituye el espacio en el que un ataque puede
ocultarse, **el margen del adversario no es constante: depende de la hora**.

Un mismo desplazamiento resulta evidentemente anómalo de madrugada e
indistinguible del ruido por la tarde.

Esto refina el hallazgo principal de la Capa 1 —la detección se degrada al
aumentar la variabilidad legítima— aplicándolo ahora dentro de un mismo
dispositivo, en función de la franja horaria.

### Ventana óptima de ataque: 15–17 h

En ese intervalo coinciden las dos condiciones que favorecen al adversario:

1. **Máxima variabilidad legítima** (std entre 27.01 y 28.75)
2. **Máxima proximidad a los umbrales de decisión** (media entre 69 y 83 ppb,
   con τ=58 por debajo y τ=90 por encima)

No hay que elegir entre ocultamiento y daño: ambas se maximizan en la misma
franja.

### Techo nocturno: restricción estructural sobre el adversario

Entre las 22 h y las 8 h, el máximo histórico registrado en 2025 no supera
los 61 ppb; a las 7 h el máximo es de 32 ppb.

**La banda "Mala" (>90 ppb) es inalcanzable de noche por medios legítimos.**
No ocurrió una sola vez en todo el año.

Consecuencia directa: un ataque que sostenga valores altos durante la noche
produce una configuración sin precedente en el registro histórico. Su
detección no requiere un modelo complejo — basta un modelo condicionado por
hora del día.

De aquí se deriva la restricción operativa del adversario: **para permanecer
indetectable, un ataque sostenido debe respetar la forma del ciclo diurno.**
Puede desplazar la curva, pero no deformarla. Esta restricción acota el
desplazamiento acumulado alcanzable y es cuantificable.

### Observación pendiente de verificación

Las horas 1–4 registran medias superiores (13–16 ppb) a las horas 6–8
(6–10 ppb). El mínimo diario se sitúa a las 7 h, no a medianoche.

Hipótesis: ozono residual de la tarde anterior, aún no consumido por
titulación con NO. Requiere verificación antes de afirmarse.